<a href="https://colab.research.google.com/github/ali-vosoughi/Large-scale-nonlinear-causality/blob/main/Demo_lsNGC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo for lsNGC


This notebook reproduces the bundled three-node logistic demonstration from the reference repository.


Input arrays have shape `(N, T)`: one row per variable and one column per time sample. The returned affinity matrix uses source rows and target columns. We call `lsNGC` as `granger` below and retain the original benchmark settings and AUROC calculation.

Run all cells from a local checkout or in Colab. The setup cell clones the repository only when a checkout is not already present; package installation is skipped when `lsngc` is installed and importable.


In [1]:
from pathlib import Path
import importlib.metadata
import os
import subprocess
import sys

repository = Path.cwd()
if not (repository / "pyproject.toml").is_file():
    repository = repository / "Large-scale-nonlinear-causality"
    if not repository.exists():
        subprocess.run([
            "git", "clone",
            "https://github.com/ali-vosoughi/Large-scale-nonlinear-causality.git",
            str(repository),
        ], check=True)
    os.chdir(repository)


In [2]:
try:
    importlib.metadata.version("lsngc")
    from lsngc import lsNGC
except (ImportError, importlib.metadata.PackageNotFoundError):
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-e", ".[demo]"
    ], check=True)


In [3]:
print("Repository:", Path.cwd())


Repository: C:\Users\abc\projects\lsngc


In [4]:
import numpy as np
from lsngc import lsNGC as granger
from lsngc.evaluate import recovery_performance
from lsngc.simulate import load_logistic3


## Synthetic benchmark from Scientific Reports 11, 7817 (2021)

[Paper and Methods](https://doi.org/10.1038/s41598-021-87316-6) | [Paper page](https://ali-vosoughi.github.io/publications/lsngc/)


In [5]:
benchmark_dataset=['logistic3']
benchmark_titles={'logistic3':'3-Fan In'}
benchmark_TS_set=np.array([50,100,200,500])
bechmark_algorithms=[]

algorithms_title={}
bechmark_algorithms_title=[algorithms_title[i] for i in bechmark_algorithms]

benchmark_label=dict()
AUC=dict()

### Bundled logistic three-node dataset

- The file contains 100 realizations, each with 3 nodes and 500 samples.
- This demonstration retains the first 50 realizations and the final 50, 100, 200, or 500 samples of each.
- The demonstration uses lag order 1, `k_f=2`, `k_g=2`, normalization, and the original k-means seed 123.
- The paper's broader experiments use additional datasets and parameter selection described in its Methods and supplement; this is the repository demonstration.


### Recover the directed network

AUROC is computed from the continuous affinity scores using the original convention, including matrix diagonals.


In [6]:
for TS in benchmark_TS_set:
    examples = [load_logistic3(realization=i, n_samples=int(TS)) for i in range(50)]
    benchmark_label['logistic3'] = [truth for _, truth in examples]
    ts_logistic = [series for series, _ in examples]
    Adj_lsgc = [
        granger(series, k_f=2, k_g=2, ar_order=1, normalize=1, seed=123)[0]
        for series in ts_logistic
    ]
    AUC['granger_logistic3_' + str(TS)] = recovery_performance(
        Adj_lsgc, benchmark_label['logistic3']
    )
    print("logistic3, auc lsgc for TS=" + str(TS) + ":",
          AUC['granger_logistic3_' + str(TS)].mean(),
          "+", AUC['granger_logistic3_' + str(TS)].std())


logistic3, auc lsgc for TS=50: 1.0 + 0.0


logistic3, auc lsgc for TS=100: 1.0 + 0.0


logistic3, auc lsgc for TS=200: 1.0 + 0.0


logistic3, auc lsgc for TS=500: 1.0 + 0.0
